# Entity Annotation Statistics

Extracts and counts all NER annotations from the `generated_text` field across all JSON files in `generated_text/`.  
Annotation format: `[ENTITY_NAME = value unit]` — entity name is extracted between `[` and ` =`.

In [1]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data_dir = Path("../generated_text")
json_files = sorted(data_dir.glob("*.json"))
print(f"Files found: {len(json_files)}")

Files found: 7281


In [3]:
# Pattern captures entity name: text between '[' and ' ='
pattern = re.compile(r'\[([^\[\]=]+?) =')

entity_counter = Counter()      # total occurrences across all files
entity_file_counter = Counter() # number of files containing each entity
missing_generated_text = []

for path in json_files:
    data = json.loads(path.read_text(encoding="utf-8"))
    text = data.get("generated_text", "")
    if not text:
        missing_generated_text.append(path.name)
        continue
    entities = pattern.findall(text)
    entity_counter.update(entities)
    entity_file_counter.update(set(entities))  # count each entity once per file

print(f"Distinct entity types found : {len(entity_counter)}")
print(f"Total annotations           : {sum(entity_counter.values())}")
if missing_generated_text:
    print(f"Files without generated_text: {len(missing_generated_text)}")

Distinct entity types found : 374
Total annotations           : 98677


In [4]:
df = pd.DataFrame({
    "entity": list(entity_counter.keys()),
    "total_occurrences": list(entity_counter.values()),
    "files_containing": [entity_file_counter[e] for e in entity_counter.keys()],
}).sort_values("total_occurrences", ascending=False).reset_index(drop=True)

top40 = df.head(40)
top40

,entity,total_occurrences,files_containing
0,T,27837,7241
1,Wwi,8625,6595
2,am,7957,7150
3,Wwb,6341,5934
4,N_R_i,5891,5530
5,Li,5537,4332
6,F_R_i,5019,4821
7,Lp,4887,3899
8,tp,4647,3957
9,ab,4438,4108


In [5]:
import ast, operator

def safe_eval(expr):
    try:
        return float(expr)
    except (ValueError, TypeError):
        try:
            tree = ast.parse(str(expr), mode="eval")
            return _eval(tree.body)
        except Exception:
            return None

def _eval(node):
    ops = {ast.Add: operator.add, ast.Sub: operator.sub,
           ast.Mult: operator.mul, ast.Div: operator.truediv}
    if isinstance(node, ast.Constant):
        return float(node.value)
    if isinstance(node, ast.BinOp):
        return ops[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -_eval(node.operand)
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")

# Build full comparison table for all classes
species_dir = Path("../species_data")
class_vals = {}

for species_path in sorted(species_dir.glob("*.json")):
    species_data = json.loads(species_path.read_text(encoding="utf-8"))
    tax = species_data.get("taxonomy", {})
    tax_class = tax.get("class", "Unknown")

    if tax_class not in class_vals:
        class_vals[tax_class] = {"ap": set(), "tp": set(), "tR": set(),
                                  "ap_n": 0, "tp_n": 0, "tR_n": 0}

    for e in species_data.get("data", []):
        ent = e.get("entity")
        if ent not in ("ap", "tp", "tR") or e.get("entity.value") is None:
            continue
        val = safe_eval(e["entity.value"])
        if val is None:
            continue
        class_vals[tax_class][ent].add(val)
        class_vals[tax_class][f"{ent}_n"] += 1

summary_rows = []
for cls, d in class_vals.items():
    row = {"class": cls}
    for e in ("ap", "tp", "tR"):
        row[f"n_species ({e})"] = d[f"{e}_n"] if d[f"{e}_n"] > 0 else "-"
    for a, b in [("ap", "tp"), ("ap", "tR"), ("tp", "tR")]:
        if d[f"{a}_n"] == 0 or d[f"{b}_n"] == 0:
            row[f"n ({a}=={b})"] = "-"
            row[f"n ({a}!={b})"] = "-"
        else:
            eq  = sum(1 for v in d[a] if v in d[b])
            neq = sum(1 for v in d[a] if v not in d[b])
            row[f"n ({a}=={b})"] = eq
            row[f"n ({a}!={b})"] = neq
    if all(d[f"{e}_n"] > 0 for e in ("ap", "tp", "tR")):
        row["n (ap==tp==tR)"] = len(d["ap"] & d["tp"] & d["tR"])
    else:
        row["n (ap==tp==tR)"] = "-"
    summary_rows.append(row)

result_df = (
    pd.DataFrame(summary_rows)
    .sort_values("class")
    .reset_index(drop=True)
)
print(f"Total classes: {len(result_df)}")
result_df


Total classes: 54


,class,n_species (ap),n_species (tp),n_species (tR),n (ap==tp),n (ap!=tp),n (ap==tR),n (ap!=tR),n (tp==tR),n (tp!=tR),n (ap==tp==tR)
0,Acoelomorpha,-,1,-,-,-,-,-,-,-,-
1,Actinopterygii,107,309,3,24,30,0,54,2,115,0
2,Amphibia,8,233,1,4,2,0,6,0,43,0
3,Anopla,-,1,-,-,-,-,-,-,-,-
4,Anthozoa,-,1,-,-,-,-,-,-,-,-
5,Appendicularia,-,-,-,-,-,-,-,-,-,-
6,Arachnida,-,6,-,-,-,-,-,-,-,-
7,Archiacanthocephala,-,1,-,-,-,-,-,-,-,-
8,Ascidiacea,2,4,-,0,2,-,-,-,-,-
9,Asteroidea,-,10,-,-,-,-,-,-,-,-


In [6]:
# Classification of all 54 classes into 5 groups

has_ap = result_df["n_species (ap)"] != "-"
has_tp = result_df["n_species (tp)"] != "-"
has_tR = result_df["n_species (tR)"] != "-"
has_none = ~has_ap & ~has_tp & ~has_tR

only_ap   = result_df[ has_ap & ~has_tp & ~has_tR][["class", "n_species (ap)"]].reset_index(drop=True)
only_tp   = result_df[~has_ap &  has_tp & ~has_tR][["class", "n_species (tp)"]].reset_index(drop=True)
only_tR   = result_df[~has_ap & ~has_tp &  has_tR][["class", "n_species (tR)"]].reset_index(drop=True)
no_entity = result_df[has_none][["class"]].reset_index(drop=True)
remaining = result_df[~(
    (has_ap & ~has_tp & ~has_tR) |
    (~has_ap & has_tp & ~has_tR) |
    (~has_ap & ~has_tp & has_tR) |
    has_none
)][["class", "n_species (ap)", "n_species (tp)", "n_species (tR)"]].reset_index(drop=True)

print(f"── Classes with ONLY ap ({len(only_ap)}) ──")
print(only_ap.to_string(index=False))

print(f"\n── Classes with ONLY tp ({len(only_tp)}) ──")
print(only_tp.to_string(index=False))

print(f"\n── Classes with ONLY tR ({len(only_tR)}) ──")
print(only_tR.to_string(index=False) if len(only_tR) else "  (none)")

print(f"\n── Classes with NO entity among ap/tp/tR ({len(no_entity)}) ──")
print(no_entity.to_string(index=False))

print(f"\n── Remaining classes ({len(remaining)}) — have 2 or 3 of ap/tp/tR ──")
print(remaining.to_string(index=False))

print(f"\nTotal: {len(only_ap)+len(only_tp)+len(only_tR)+len(no_entity)+len(remaining)} classes")


── Classes with ONLY ap (2) ──
      class n_species (ap)
Leptocardii              1
Monogononta              1

── Classes with ONLY tp (30) ──
              class n_species (tp)
       Acoelomorpha              1
             Anopla              1
           Anthozoa              1
          Arachnida              6
Archiacanthocephala              1
         Asteroidea             10
       Branchiopoda             90
        Cephalopoda             16
        Chromadorea              2
         Clitellata             12
         Entognatha              5
       Eutardigrada              4
       Gastrotricha              1
       Gymnolaemata              2
        Hexanauplia              6
      Holothuroidea              3
           Hydrozoa              1
      Ichthyostraca              1
           Mammalia           1435
          Monogenea              1
               Nuda              1
          Ostracoda              1
          Phoronida              2
     Polyplacop

In [7]:
# Save the 5 class groups to a JSON file — class names only
output = {
    "only_ap":   only_ap["class"].tolist(),
    "only_tp":   only_tp["class"].tolist(),
    "only_tR":   only_tR["class"].tolist(),
    "no_entity": no_entity["class"].tolist(),
    "remaining": remaining["class"].tolist(),
}

out_path = Path("class_entity_groups.json")
out_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved to {out_path}")
for k, v in output.items():
    print(f"  {k:<12}: {len(v)} classes")


Saved to class_entity_groups.json
  only_ap     : 2 classes
  only_tp     : 30 classes
  only_tR     : 0 classes
  no_entity   : 5 classes
  remaining   : 17 classes


In [8]:
# Per-species analysis for all 17 remaining classes
# For each class: count species where tp==tR, tp!=tR, ap==tp, ap!=tp, ap==tR, ap!=tR
# (only pairs where both entities exist in the same species file)

species_dir = Path("../species_data")
remaining_classes = set(remaining["class"].tolist())

class_species = {}  # class -> list of {species, ap, tp, tR}

for species_path in sorted(species_dir.glob("*.json")):
    species_data = json.loads(species_path.read_text(encoding="utf-8"))
    tax = species_data.get("taxonomy", {})
    tax_class = tax.get("class", "Unknown")
    if tax_class not in remaining_classes:
        continue

    species_name = tax.get("specie") or tax.get("species") or species_path.stem
    values = {}
    for e in species_data.get("data", []):
        if e.get("entity") in ("ap", "tp", "tR") and e.get("entity.value") is not None:
            val = safe_eval(e["entity.value"])
            if val is not None:
                values[e["entity"]] = val

    class_species.setdefault(tax_class, []).append({
        "species": species_name,
        "ap": values.get("ap"),
        "tp": values.get("tp"),
        "tR": values.get("tR"),
    })

# Build summary per class
summary_rows = []
for cls in sorted(remaining_classes):
    rows = class_species.get(cls, [])
    df_cls = pd.DataFrame(rows)

    def pair_counts(df, a, b):
        valid = df.dropna(subset=[a, b])
        eq = (valid[a] == valid[b]).sum()
        diff = (valid[a] != valid[b]).sum()
        return int(eq), int(diff)

    tp_tR_eq,  tp_tR_diff  = pair_counts(df_cls, "tp", "tR")
    ap_tp_eq,  ap_tp_diff  = pair_counts(df_cls, "ap", "tp")
    ap_tR_eq,  ap_tR_diff  = pair_counts(df_cls, "ap", "tR")

    triple = df_cls.dropna(subset=["ap", "tp", "tR"])
    ap_tp_tR_eq = int(((triple["ap"] == triple["tp"]) & (triple["tp"] == triple["tR"])).sum())

    summary_rows.append({
        "class":          cls,
        "n(ap)":          df_cls["ap"].notna().sum(),
        "n(tp)":          df_cls["tp"].notna().sum(),
        "n(tR)":          df_cls["tR"].notna().sum(),
        "tp==tR":         tp_tR_eq  or "-" if tp_tR_eq + tp_tR_diff == 0 else tp_tR_eq,
        "tp!=tR":         tp_tR_diff or "-" if tp_tR_eq + tp_tR_diff == 0 else tp_tR_diff,
        "ap==tp":         ap_tp_eq  or "-" if ap_tp_eq + ap_tp_diff == 0 else ap_tp_eq,
        "ap!=tp":         ap_tp_diff or "-" if ap_tp_eq + ap_tp_diff == 0 else ap_tp_diff,
        "ap==tR":         ap_tR_eq  or "-" if ap_tR_eq + ap_tR_diff == 0 else ap_tR_eq,
        "ap!=tR":         ap_tR_diff or "-" if ap_tR_eq + ap_tR_diff == 0 else ap_tR_diff,
        "ap==tp==tR":     ap_tp_tR_eq if len(triple) > 0 else "-",
    })

all_classes_df = pd.DataFrame(summary_rows)

# Replace 0 counts for pairs that don't exist with "-"
for col in ["tp==tR","tp!=tR","ap==tp","ap!=tp","ap==tR","ap!=tR","ap==tp==tR"]:
    all_classes_df[col] = all_classes_df[col].apply(lambda x: "-" if x == 0 and isinstance(x, int) else x)

print(f"Per-species comparison for all {len(all_classes_df)} remaining classes:\n")
all_classes_df


Per-species comparison for all 17 remaining classes:



,class,n(ap),n(tp),n(tR),tp==tR,tp!=tR,ap==tp,ap!=tp,ap==tR,ap!=tR,ap==tp==tR
0,Actinopterygii,102,259,1,-,1,-,-,-,-,-
1,Amphibia,8,188,1,-,-,-,-,-,-,-
2,Ascidiacea,2,4,0,-,-,-,-,-,-,-
3,Aves,0,1744,1744,1,1743,-,-,-,-,-
4,Bivalvia,8,32,0,-,-,-,-,-,-,-
5,Cephalaspidomorphi,2,4,0,-,-,-,-,-,-,-
6,Chondrichthyes,17,44,0,-,-,-,-,-,-,-
7,Echinoidea,1,7,0,-,-,-,-,-,-,-
8,Gastropoda,1,33,0,-,-,-,-,-,-,-
9,Insecta,3,9,0,-,-,-,-,-,-,-


In [9]:
# Generate PNG for per-species analysis of all 17 remaining classes
col_labels = ["class", "n(ap)", "n(tp)", "n(tR)",
              "tp==tR", "tp!=tR",
              "ap==tp", "ap!=tp",
              "ap==tR", "ap!=tR",
              "ap==tp==tR"]

data = all_classes_df[col_labels].values.tolist()
n_rows = len(data)

fig = plt.figure(figsize=(80, n_rows * 1.8 + 5))

title_ax = fig.add_axes([0, 0.93, 1, 0.07])
title_ax.axis("off")
title_ax.text(0.5, 0.5, "Per-species analysis — 17 remaining classes (ap / tp / tR)",
              ha="center", va="center", fontsize=36, fontweight="bold")

table_ax = fig.add_axes([0, 0, 1, 0.91])
table_ax.axis("off")

tbl = table_ax.table(
    cellText=data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(80)

col_width_class = 0.22
col_width_other = 0.1
for (row, col), cell in tbl.get_celld().items():
    cell.set_height(0.09)
    cell.set_width(col_width_class if col == 0 else col_width_other)
    if row == 0:
        cell.set_facecolor("#1565C0")
        cell.set_text_props(color="white", fontweight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#E3F2FD")
    else:
        cell.set_facecolor("#FFFFFF")
    cell.set_edgecolor("#90CAF9")

plt.savefig("per_species_17_classes.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved to per_species_17_classes.png")


Saved to per_species_17_classes.png
